# §4.2 — Morphological Typology

Hierarchical clustering (initial 3-type view) + K-means (final 2-type classification).

**Inputs:** `data/clusters_with_metrics.csv`  
**Outputs:** `data/clusters_typed.csv`, Table 1, Figures 5, 6

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
df = pd.read_csv('../data/clusters_with_metrics.csv')
print(f'{len(df):,} clusters')

FEATURES = ['fractal_dim', 'clark_evans', 'aspect_ratio', 'alpha_compactness']
data = df[FEATURES].dropna()
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
print(f'Working on {len(data):,} clusters with complete feature set')

## Figure 5 — Hierarchical clustering dendrogram

In [ ]:
# Subsample for dendrogram visualisation
sample = data_scaled[np.random.default_rng(42).choice(len(data_scaled), size=min(2000, len(data_scaled)), replace=False)]
Z = linkage(sample, method='ward')

corr_mat = data.corr()
Z_feat = linkage(corr_mat.values, method='average')

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=150)
dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=30,
           color_threshold=Z[-3, 2], above_threshold_color='grey')
axes[0].set_title('Cluster sample dendrogram (Ward)', fontsize=12)
axes[0].set_xlabel('Cluster index', fontsize=10)
axes[0].set_ylabel('Distance', fontsize=10)

sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=axes[1], linewidths=0.5)
dendrogram(Z_feat, ax=axes[1], orientation='left', no_labels=False, color_threshold=0)
axes[1].set_title('Feature correlation + dendrogram', fontsize=12)

plt.tight_layout()
plt.savefig('../figures/fig5_dendrogram.png', dpi=300, bbox_inches='tight')
plt.show()

## K-means validation (elbow + silhouette)

In [ ]:
k_range = range(2, 8)
inertia, sil, ch, db = [], [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(data_scaled)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(data_scaled, labels))
    ch.append(calinski_harabasz_score(data_scaled, labels))
    db.append(davies_bouldin_score(data_scaled, labels))
    print(f'k={k}  sil={sil[-1]:.3f}  CH={ch[-1]:.0f}  DB={db[-1]:.3f}')

## Final K-means with k=2

In [ ]:
km2 = KMeans(n_clusters=2, random_state=42, n_init=10)
df.loc[data.index, 'morph_type'] = km2.fit_predict(data_scaled)
df['morph_type'] = df['morph_type'].map({0: 'Type0', 1: 'Type1'})

# Table 1
table1 = df.groupby('morph_type')[FEATURES + ['cluster_id']].agg(
    {f: ['mean', 'std'] for f in FEATURES} | {'cluster_id': 'count'}
)
print('\n--- Table 1 ---')
print(table1)
print(f"\nType0: {(df.morph_type=='Type0').sum():,}  (expect ~5,493)")
print(f"Type1: {(df.morph_type=='Type1').sum():,}  (expect ~9,089)")

df.to_csv('../data/clusters_typed.csv', index=False)
print('\nSaved → data/clusters_typed.csv')

## Figure 6 — Spatial distribution of morphological types

In [ ]:
GRID = 1000   # 1 km grid cells
typed = df.dropna(subset=['morph_type', 'cx', 'cy'])
typed = typed.copy()
typed['gx'] = (typed['cx'] // GRID).astype(int)
typed['gy'] = (typed['cy'] // GRID).astype(int)

cell_counts = typed.groupby(['gx', 'gy', 'morph_type']).size().unstack(fill_value=0)
cell_total = cell_counts.sum(axis=1)
cell_counts = cell_counts[cell_total >= 3]   # >=3 clusters per cell
dominant = cell_counts.idxmax(axis=1)

colour = {'Type0': '#8B4513', 'Type1': '#4169E1'}
fig, ax = plt.subplots(figsize=(10, 9), dpi=150)
for (gx, gy), dom in dominant.items():
    rect = plt.Rectangle((gx * GRID, gy * GRID), GRID, GRID,
                          color=colour[dom], alpha=0.4)
    ax.add_patch(rect)

for t, marker, c in [('Type0', '^', '#8B4513'), ('Type1', 'o', '#4169E1')]:
    sub = typed[typed.morph_type == t]
    ax.scatter(sub['cx'], sub['cy'], marker=marker, s=4, alpha=0.5, color=c,
               label=t + (' (Pest/Fire)' if t == 'Type0' else ' (Uniform/Scattered)'))

ax.set_xlabel('Easting (m)', fontsize=11)
ax.set_ylabel('Northing (m)', fontsize=11)
ax.set_title('Dominant morphological type per 1 km² grid cell', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../figures/fig6_typology_map.png', dpi=300, bbox_inches='tight')
plt.show()